In [1]:
import pandas as pd

file = "data/macrotopics_pred.csv"
df = pd.read_csv(file)

In [2]:
len(df)

673587

In [3]:
df.columns

Index(['video_id', 'text', 'channel_title', 'published_at', 'view_count',
       'like_count', 'comment_count', 'occurrences', 'toxicity',
       'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack',
       'perspective_insult', 'perspective_severe_toxicity',
       'perspective_obscene', 'perspective_toxicity',
       'perspective_identity_attack', 'perspective_threat', 'macrotopic_pred',
       'macrotopic_prob'],
      dtype='object')

In [3]:
# Filtra onde a quantidade de palavras é maior que 128
videos_longos = df[df['text'].str.split().str.len() > 128]

# Conta o resultado
quantidade = len(videos_longos)
print(f"Vídeos com mais de 128 palavras: {quantidade}")

Vídeos com mais de 128 palavras: 75554


In [4]:
# Filtra onde a quantidade de caracteres é maior que 128
videos_longos = df[df['text'].str.len() > 128]

# Conta o resultado
quantidade = len(videos_longos)
print(f"Vídeos com mais de 128 caracteres: {quantidade}")

Vídeos com mais de 128 caracteres: 498293


In [5]:
from transformers import AutoTokenizer

# 1. Carrega o Tokenizador (precisa ser o mesmo nome do modelo que você vai usar)
nome_modelo = "sentence-transformers/paraphrase-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)

# 2. Define uma função para contar tokens
def calcular_tokens(texto):
    # Converte para string para evitar erros caso haja valores nulos (NaN)
    texto = str(texto)
    
    # tokenizer.encode transforma o texto em lista de IDs
    # add_special_tokens=True é CRUCIAL, pois conta o [CLS] e [SEP] que o modelo usa
    tokens = tokenizer.encode(texto, add_special_tokens=True)
    return len(tokens)

# 3. Aplica a função em todo o DataFrame
# (Atenção: Isso é mais lento que o split() normal porque faz processamento real)
df['n_tokens'] = df['text'].apply(calcular_tokens)

# 4. Filtra onde a quantidade de TOKENS é maior que 128
videos_estourados = df[df['n_tokens'] > 128]

# Conta o resultado
quantidade = len(videos_estourados)
print(f"Vídeos com mais de 128 tokens: {quantidade}")

# Opcional: Ver a diferença média
print(f"Média de tokens por vídeo: {df['n_tokens'].mean():.2f}")

/home/uselection/miniconda3/envs/rapids-24.08/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (665 > 512). Running this sequence through the model will result in indexing errors


Vídeos com mais de 128 tokens: 125313
Média de tokens por vídeo: 82.38


In [6]:
import pandas as pd

# 1. Definir as faixas (BINS)
# O último valor é float('inf') para pegar tudo que for gigante
# Usamos intervalos comuns em NLP: 0-32, 32-64, 64-128, etc.
bins = [0, 32, 64, 128, 256, 512, float('inf')]

# 2. Definir os rótulos (LABELS) para cada faixa
labels = ['0 - 32', '33 - 64', '65 - 128', '129 - 256', '257 - 512', '> 512']

# 3. Criar a coluna de categorização
# right=True inclui o número da direita (ex: se tiver 128, cai na faixa 65-128)
df['faixa_tokens'] = pd.cut(df['n_tokens'], bins=bins, labels=labels, right=True)

# 4. Contar a quantidade em cada faixa
distribuicao = df['faixa_tokens'].value_counts().sort_index()

# 5. Criar o DataFrame final bonitinho
df_resultado = pd.DataFrame({
    'Faixa de Tokens': distribuicao.index,
    'Qtd Vídeos': distribuicao.values
})

# (Opcional) Adicionar coluna de porcentagem para ter noção do impacto
df_resultado['% do Total'] = (df_resultado['Qtd Vídeos'] / len(df) * 100).round(2)

# Exibir
print("Distribuição de Tokens no Dataset:")
display(df_resultado)

# 6. Dica Visual: Saber exatamente quantos você perde cortando em 128
perda = df[df['n_tokens'] > 128].shape[0]
print(f"\nResumo Crítico:")
print(f"Se cortar em 128 tokens, {perda} vídeos ({perda/len(df):.1%}) serão truncados.")

Distribuição de Tokens no Dataset:


,Faixa de Tokens,Qtd Vídeos,% do Total
0,0 - 32,223529,33.18
1,33 - 64,158989,23.60
2,65 - 128,165756,24.61
3,129 - 256,89784,13.33
4,257 - 512,31881,4.73
5,> 512,3648,0.54



Resumo Crítico:
Se cortar em 128 tokens, 125313 vídeos (18.6%) serão truncados.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/preprocessed_english_titles.csv")

In [3]:
df.head()

,Unnamed: 0,video_id,title,description,title_description,lang,clean_text
0,0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Free bus travel for migrants scrapped. For 5 m...,en,free travel migrant scrap minute
1,1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",What is Spiritual Warfare? This charge I commi...,en,spiritual warfare charge commit unto thee timo...
2,2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...","80 Putins Have Layers In February 2024, Tucker...",en,putin layer february tucker carlson stand onio...
3,3,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,en,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...
4,4,AKU0RokegSo,Los Angeles rain: Studio City homes evacuated ...,An atmospheric river storm has caused mudslide...,Los Angeles rain: Studio City homes evacuated ...,en,angeles rain studio city home evacuate mudslid...
